In [1]:
import os, warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn


# Configuration
START, END  = "1995-06-01", "2025-12-01"
TEST_MONTHS = 60
HORIZONS    = [1, 3, 6, 12]
WINDOW      = 12            # months of history fed to the model
H_HID, G_HID = 16, 24      # small spatial / GRU hidden sizes
EPOCHS, PATIENCE, LR, WD, DROP = 400, 30, 5e-3, 1e-4, 0.2
SEEDS       = [0, 1, 2]    # mini-ensemble to reduce run-to-run variance
OUTPUT_DIR  = "Code Outputs/GNN Outputs"; os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)
def rmse(p, a): p, a = np.asarray(p, float), np.asarray(a, float); return np.sqrt(np.nanmean((p - a) ** 2))
def mae(p, a):  p, a = np.asarray(p, float), np.asarray(a, float); return np.nanmean(np.abs(p - a))

# DATA: levels -> differences ; node features ; adjacency

lev = pd.read_excel("Code Outputs/Gap Interpolation Outputs/Unified_Interpolated_Levels.xlsx")
lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
        .sort_index().asfreq("MS").loc[START:END])
LAKES = list(L.columns); N = len(LAKES)
Lv = L.values                                   # (T, N) levels
dL = np.diff(Lv, axis=0)                         # (T-1, N) monthly changes
dates = L.index[1:]                              # dates aligned to dL rows
T = len(dL)
split = T - TEST_MONTHS                           # first test target row (in dL index)

# adjacency from EDA residual correlation (fallback: correlation of the diffs)
adj_path = "Code Outputs/EDA Outputs/EDA_06_corr_residual.csv"
if os.path.exists(adj_path):
    C = pd.read_csv(adj_path, index_col=0).reindex(index=LAKES, columns=LAKES).values
else:
    C = np.corrcoef(dL[:split].T)
A = np.abs(C); A[A < 0.2] = 0.0                  # keep meaningful edges
A = A + np.eye(N)                                # self-loops
d = A.sum(1); Dinv = np.diag(1.0 / np.sqrt(d))
A_norm = torch.tensor(Dinv @ A @ Dinv, dtype=torch.float32)   # symmetric-normalised

# standardise the diff feature using TRAIN rows only
mu, sd = dL[:split].mean(0), dL[:split].std(0) + 1e-8
dstd = ((dL - mu) / sd).astype(np.float32)        # (T, N) standardised diffs (float32)
month = dates.month.values
sinm, cosm = np.sin(2*np.pi*month/12), np.cos(2*np.pi*month/12)
# node feature tensor (T, N, F): [diff_std, sin, cos]
feats = np.stack([dstd,
                  np.repeat(sinm[:, None], N, 1),
                  np.repeat(cosm[:, None], N, 1)], axis=-1).astype(np.float32)
F_IN = feats.shape[-1]

# windowed supervised samples: X=(L past feats), y=next standardised diff
def make_windows(lo, hi):
    X, Y = [], []
    for i in range(max(WINDOW, lo), hi):
        X.append(feats[i-WINDOW:i]); Y.append(dstd[i])
    return np.array(X), np.array(Y)
val_start = int(split * 0.85)
Xtr, Ytr = make_windows(WINDOW, val_start)
Xva, Yva = make_windows(val_start, split)

# MODEL: GCN (spatial mix) x2  ->  GRU (temporal, shared across nodes)

class GCN(nn.Module):
    def __init__(s, fin, fout): super().__init__(); s.lin = nn.Linear(fin, fout)
    def forward(s, x, An):                        # x: (B, N, Fin)
        x = s.lin(x)
        x = torch.einsum("ij,bjf->bif", An, x)     # neighbour aggregation A @ x
        return torch.relu(x)

class STGNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.g1 = GCN(F_IN, H_HID); s.g2 = GCN(H_HID, H_HID)
        s.gru = nn.GRU(H_HID, G_HID, batch_first=True)
        s.drop = nn.Dropout(DROP); s.out = nn.Linear(G_HID, 1)
    def forward(s, x, An):                          # x: (B, L, N, Fin)
        B, Lw, n, f = x.shape
        h = x.reshape(B*Lw, n, f)
        h = s.g2(s.g1(h, An), An)                   # (B*Lw, N, H_HID)
        h = h.reshape(B, Lw, n, -1).permute(0, 2, 1, 3).reshape(B*n, Lw, -1)
        _, hn = s.gru(h)                            # (1, B*n, G_HID)
        o = s.out(s.drop(hn[-1]))                   # (B*n, 1)
        return o.reshape(B, n)                      # predicted next std diff per node

def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m = STGNN(); opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()
    xtr, ytr = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Ytr, dtype=torch.float32)
    xva, yva = torch.tensor(Xva, dtype=torch.float32), torch.tensor(Yva, dtype=torch.float32)
    best, best_state, wait = np.inf, None, 0
    for ep in range(EPOCHS):
        m.train(); opt.zero_grad()
        loss = lossf(m(xtr, A_norm), ytr); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad(): vl = lossf(m(xva, A_norm), yva).item()
        if vl < best - 1e-5: best, best_state, wait = vl, {k: v.clone() for k, v in m.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    m.load_state_dict(best_state); m.eval()
    return m

# TRAIN ENSEMBLE + ROLLING RECURSIVE FORECAST over the test period

models = [train_one(s) for s in SEEDS]
print(f"trained {len(models)} GNN seeds (window={WINDOW}, hidden={H_HID}/{G_HID})")

def predict_diff(window_feats):                     # window_feats: (L, N, F) numpy
    x = torch.tensor(window_feats[None], dtype=torch.float32)
    with torch.no_grad():
        p = np.mean([mm(x, A_norm).numpy()[0] for mm in models], axis=0)  # ensemble mean, std diff
    return p                                         # (N,) standardised diff

preds = {}
for o in range(split, T):                            # o = target row in dL (know feats[:o])
    win = feats[o-WINDOW:o].copy()                   # observed window
    base = Lv[o]                                     # level at dL-row o corresponds to L[o] ... see note

    cum = 0.0
    Hh = min(max(HORIZONS), T - o)
    for k in range(Hh):
        pstd = predict_diff(win)                      # (N,) standardised diff
        pdiff = pstd * sd + mu                        # de-standardise -> real diff
        cum = cum + pdiff
        h = k + 1
        if h in HORIZONS:
            for j, lk in enumerate(LAKES):
                preds[("GNN", lk, o + k, h)] = base[j] + cum[j]   # level forecast
        # roll the window: append predicted std diff with the NEXT month's season
        nxt = o + k
        mth = dates[nxt].month if nxt < T else ((dates[-1].month % 12) + 1)
        newf = np.stack([pstd,
                         np.full(N, np.sin(2*np.pi*mth/12), np.float32),
                         np.full(N, np.cos(2*np.pi*mth/12), np.float32)], axis=-1)
        win = np.concatenate([win[1:], newf[None]], axis=0)

# Score  (target row o in dL -> level index o+1)

rows = []
for lk in LAKES:
    j = LAKES.index(lk)
    for h in HORIZONS:
        P, A_ = [], []
        for o in range(split, T):
            key = ("GNN", lk, o + h - 1, h)
            if key in preds and (o + h) < len(Lv):
                P.append(preds[key]); A_.append(Lv[o + h][j])
        rows.append({"Lake": lk, "Model": "GNN", "Horizon_m": h,
                     "RMSE_m": round(rmse(P, A_), 4), "MAE_m": round(mae(P, A_), 4)})
gnn = pd.DataFrame(rows)


# Combine with FC4 (SARIMA/VAR/RandomWalk) + skill vs SARIMA

fc4 = pd.read_csv("Code Outputs/Forecast Var Outputs/FC4_metrics.csv")
allm = pd.concat([fc4[["Lake", "Model", "Horizon_m", "RMSE_m", "MAE_m"]], gnn], ignore_index=True)
sar = allm[allm.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]
allm["skill_vs_SARIMA_%"] = allm.apply(
    lambda r: round(100 * (sar[(r.Lake, r.Horizon_m)] - r.RMSE_m) / sar[(r.Lake, r.Horizon_m)], 1), axis=1)
allm.to_csv(out("FC6_gnn_metrics.csv"), index=False)

print("\n=== GNN skill vs SARIMA (%) by horizon ===")
print(allm[allm.Model == "GNN"].pivot(index="Lake", columns="Horizon_m", values="skill_vs_SARIMA_%").to_string())
print("\n=== mean skill vs SARIMA by model x horizon ===")
print(allm.pivot_table(index="Model", columns="Horizon_m", values="skill_vs_SARIMA_%").round(1).to_string())

order = ["RandomWalk", "SARIMA", "VAR", "GNN"]
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, h in zip(axes, [1, 6]):
    sub = allm[(allm.Horizon_m == h) & (allm.Model.isin(order))].pivot(index="Lake", columns="Model", values="RMSE_m")[order]
    sub.plot(kind="bar", ax=ax); ax.set_title(f"RMSE at horizon {h} months", fontweight="bold")
    ax.set_ylabel("RMSE (m)"); ax.tick_params(axis="x", rotation=45)
plt.suptitle("Does the spatio-temporal GNN beat the linear baselines?", fontweight="bold")
plt.tight_layout(); plt.savefig(out("FC6_compare.png"), dpi=200); plt.close()
print("\nDone. Outputs in:", OUTPUT_DIR)

trained 3 GNN seeds (window=12, hidden=16/24)

=== GNN skill vs SARIMA (%) by horizon ===
Horizon_m          1     3     6     12
Lake                                   
Lake Albert     -27.3 -44.8 -33.6   8.5
Lake Edward      -6.2 -21.2  -8.0 -26.5
Lake Kivu        -3.5 -32.8 -20.6 -19.6
Lake Malawi      -5.5  -8.4 -10.5 -24.7
Lake Tanganyika -18.1 -30.2 -15.2 -22.8
Lake Turkana    -18.0 -14.3  -7.3  -8.6
Lake Victoria   -27.3 -38.0  -9.1  -2.5

=== mean skill vs SARIMA by model x horizon ===
Horizon_m     1     3     6     12
Model                             
GNN        -15.1 -27.1 -14.9 -13.7
RandomWalk -53.4 -65.8 -47.2  -0.8
SARIMA       0.0   0.0   0.0   0.0
VAR          0.5   2.7   3.3   0.3
VARX        11.7   6.0   4.1   3.0

Done. Outputs in: Code Outputs/GNN Outputs
